# USA

In [2]:
import os, sys
import json
project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
from config import DATA_DIR

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)
with open(f"{DATA_DIR}/vlm_cot_predictions.json", 'r') as f:
    cot = json.load(f)
with open(f"{DATA_DIR}/vlm_forward_pass_predictions.json", 'r') as f:
    nocot = json.load(f)
with open(f"{DATA_DIR}/gpt_predictions.json", 'r') as f:
    gpt = json.load(f)
with open(f"{DATA_DIR}/climate-answers_predictions.json", 'r') as f:
    climate = json.load(f)
with open(f"{DATA_DIR}/random_predictions.json", 'r') as f:
    random = json.load(f)

for k in cot:
    if k not in nocot:
        print(k)
for k in nocot:
    if k not in cot:
        print(k)

print(len(cot), len(nocot), len(gpt))

In [3]:
from typing import Dict, Any, Tuple, Optional

def evaluate_ordinal_results(
    data: Dict[Any, Dict[str, int]],
    *,
    invalid_value: int = -1,
    fill_value: int = 4,          # mean of 0–9
    min_rating: int = 0,
    max_rating: int = 9
) -> Dict[str, float]:
    """
    Compute:
      - count_invalid: number of answers equal to `invalid_value`
      - MSE, MAE between (answers with invalids replaced by `fill_value`) and ground truths
      - QWK (Quadratic Weighted Kappa) between those answers and ground truths

    Parameters
    ----------
    data : dict
        Mapping -> {'answer': int, 'ground_truth': int}
    invalid_value : int
        Value considered invalid in answers (default -1).
    fill_value : int
        Replacement value for invalid answers (default 4).
    min_rating, max_rating : int
        Inclusive rating bounds (default 0..9).

    Returns
    -------
    dict with keys: 'count_invalid', 'mse', 'mae', 'qwk'
    """

    # Collect aligned vectors, replacing invalid answers
    answers = []
    truths = []
    count_invalid = 0
    for _, rec in data.items():
        a = rec['answer']
        g = rec['ground_truth']
        if a == invalid_value:
            a = fill_value
            count_invalid += 1
        # (Optionally) clip to valid bounds to avoid out-of-range
        a = max(min_rating, min(max_rating, a))
        g = max(min_rating, min(max_rating, g))
        answers.append(a)
        truths.append(g)

    n = len(answers)
    if n == 0:
        return {'count_invalid': 0, 'mse': 0.0, 'mae': 0.0, 'qwk': 1.0}

    # --- MSE & MAE ---
    se = 0
    ae = 0
    for a, g in zip(answers, truths):
        d = (a - g)/10.
        se += d * d
        ae += abs(d)
    mse = se / n
    mae = ae / n

    # --- Quadratic Weighted Kappa ---
    # Ratings are integers in [min_rating, max_rating]
    k = max_rating - min_rating + 1

    # Observed matrix O[i, j]: truth=i, ans=j
    O = [[0] * k for _ in range(k)]
    hist_truth = [0] * k
    hist_ans = [0] * k

    for g, a in zip(truths, answers):
        gi = g - min_rating
        ai = a - min_rating
        O[gi][ai] += 1
        hist_truth[gi] += 1
        hist_ans[ai] += 1

    # Weight matrix W[i, j] = (i - j)^2 / (k - 1)^2
    denom = float((k - 1) ** 2) if k > 1 else 1.0

    # Expected matrix E = outer(hist_truth, hist_ans) / n
    # Compute numerator and denominator for QWK: 1 - (sum(W*O) / sum(W*E))
    import math

    weighted_obs = 0.0
    weighted_exp = 0.0
    for i in range(k):
        for j in range(k):
            w = ((i - j) ** 2) / denom
            expected = (hist_truth[i] * hist_ans[j]) / n
            weighted_obs += w * O[i][j]
            weighted_exp += w * expected

    # Guard against degenerate case where weighted_exp == 0
    qwk = 1.0 if math.isclose(weighted_exp, 0.0) else 1.0 - (weighted_obs / weighted_exp)

    return {
        'count_invalid': float(count_invalid),
        'mse': float(mse),
        'mae': float(mae),
        'qwk': float(qwk),
    }

In [4]:
import pandas as pd
pd.DataFrame([
    evaluate_ordinal_results({k: c for k, c in climate.items() if 'test' in c['img_path']}),
    evaluate_ordinal_results({k: c for k, c in cot.items() if 'test' in c['img_path']}),
    evaluate_ordinal_results({k: c for k, c in nocot.items() if 'test' in c['img_path']}),
    evaluate_ordinal_results({k: c for k, c in gpt.items() if 'test' in c['img_path']}),
    evaluate_ordinal_results({k: c for k, c in random.items() if 'test' in c['img_path']}),
], index=['climate','cot','nocot','gpt','random'])

In [12]:
import glob
import numpy as np
from tqdm import tqdm
from scipy.stats import mannwhitneyu
import os, sys
import pandas as pd
project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from config import DATA_DIR

def compute_burn_control_stats(json_path, fwi=True):
    df = pd.read_json(json_path).T

    # Accumulators
    burned_sum_all, burned_count_all = 0.0, 0
    control_sum_all, control_count_all = 0.0, 0
    burned_means_per_image = []
    control_means_per_image = []

    # 1) Burned predictions: global mean + masked/unmasked means (with masks)
    for lat_lon, row in tqdm(df[df.img_path.apply(lambda f: '/test' in f)].iterrows(), desc="Burned preds"):
        pred = float(row['answer'])
        burned_sum_all += pred
        burned_count_all += 1
        burned_means_per_image.append(pred)

    # 2) Random control predictions: global mean
    for lat_lon, row in tqdm(df[df.img_path.apply(lambda f: '/test' in f)].iterrows(), desc="Control preds"):
        pred = float(row['ground_truth'])
        control_sum_all += pred
        control_count_all += 1
        control_means_per_image.append(pred)

    # Safeguards
    if burned_count_all == 0 or control_count_all == 0:
        raise ValueError("No pixels found in one of the sets.")

    # Means
    burned_mean_all = burned_sum_all / burned_count_all
    control_mean_all = control_sum_all / control_count_all
    diff_all = burned_mean_all - control_mean_all

    # Build predictions/labels (per-image level)
    preds = np.array(burned_means_per_image, dtype=float)/10.
    labels = np.array(control_means_per_image, dtype=float)/10.

    return {
        "burned_mean_all": burned_mean_all,
        "control_mean_all": control_mean_all,
        "diff_burned_minus_control": diff_all,
        "burned_means_per_image": burned_means_per_image,
        "control_means_per_image": control_means_per_image,
        # New keys for metrics:
        "preds": preds,     # per-image mean, in [0,1]
        "labels": labels,   # 1 for burned, 0 for control
    }

import numpy as np
from itertools import combinations
from scipy.stats import mannwhitneyu

def _safe_finite(arr):
    arr = np.asarray(arr, dtype=float)
    return arr[np.isfinite(arr)]

def _extract_deltas(stats, mode):
    """
    Return a per-image delta array for a given stats dict and mode:
      - mode == "burned_control": burned − control (per image)
      - mode == "masked_unmasked": masked − unmasked (per image)

    Requires per-image arrays returned by your modified compute_burn_control_stats:
      - "burned_minus_control_per_image"  (preferred for burned_control)
        OR fallback to difference of the per-image means if both lists exist
        ("burned_means_per_image", "control_means_per_image").
      - "masked_means_per_image" & "unmasked_means_per_image" for masked_unmasked.
    """
    if mode == "burned_control":
        if "burned_minus_control_per_image" in stats:
            return _safe_finite(stats["burned_minus_control_per_image"])
        # Fallback if unmatched controls: difference of per-image means (not ideal but usable)
        if ("burned_means_per_image" in stats) and ("control_means_per_image" in stats):
            b = _safe_finite(stats["burned_means_per_image"])
            c = _safe_finite(stats["control_means_per_image"])
            # Lengths may differ if controls are unmatched; compute all-vs-all deltas via broadcasting
            # but that explodes size. Instead, compare distributions with MW directly downstream.
            # For a delta distribution fallback, we take elementwise up to min length.
            n = min(len(b), len(c))
            if n == 0:
                return np.array([], dtype=float)
            return b[:n] - c[:n]
        raise ValueError("Stats object lacks per-image fields for burned_control comparison.")
    else:
        raise ValueError(f"Unknown mode: {mode}")

def _pairwise_mw(deltas_list, alternative="two-sided"):
    """
    Given a list of 1D arrays (per model), compute pairwise Mann–Whitney U tests.
    Returns a dict-of-dicts with entries: {"U": U, "p": p, "AUC": auc, "n_x": n1, "n_y": n2}.
    AUC (probability of superiority) = U / (n1*n2).
    """
    k = len(deltas_list)
    U = [[None]*k for _ in range(k)]
    P = [[None]*k for _ in range(k)]
    A = [[None]*k for _ in range(k)]
    N1 = [[None]*k for _ in range(k)]
    N2 = [[None]*k for _ in range(k)]
    for i, j in combinations(range(k), 2):
        x = _safe_finite(deltas_list[i])
        y = _safe_finite(deltas_list[j])
        if len(x) == 0 or len(y) == 0:
            u = p = auc = np.nan
        else:
            u, p = mannwhitneyu(x, y, alternative=alternative)
            auc = u / (len(x)*len(y))
        U[i][j] = U[j][i] = u
        P[i][j] = P[j][i] = p
        A[i][j] = A[j][i] = auc
        N1[i][j] = N1[j][i] = len(x)
        N2[i][j] = N2[j][i] = len(y)
        # Diagonal
    for i in range(k):
        U[i][i] = P[i][i] = A[i][i] = 0.0
        N1[i][i] = N2[i][i] = len(_safe_finite(deltas_list[i]))
    return {"U": U, "p": P, "AUC": A, "n_x": N1, "n_y": N2}


# ----------------------- Metrics & summary table -----------------------
from scipy.stats import mannwhitneyu, pearsonr

def _roc_auc_from_scores(y: np.ndarray, p: np.ndarray) -> float:
    """ROC AUC via Mann–Whitney: P(p_pos > p_neg)."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pos = p[y == 1]
    neg = p[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    U = mannwhitneyu(pos, neg, alternative="greater", method="asymptotic").statistic
    return float(U / (len(pos) * len(neg)))

def _brier(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    return float(np.mean((p - y) ** 2))

def _logloss(y: np.ndarray, p: np.ndarray, eps: float = 1e-12) -> float:
    y = np.asarray(y>0.5, dtype=float)
    p = np.asarray(p, dtype=float)
    p = np.clip(p, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))

def _pearson_corr(y: np.ndarray, p: np.ndarray) -> float:
    # Point-biserial correlation equals Pearson between binary y and continuous p
    if len(y) < 2 or np.all(y == y[0]):
        return np.nan
    r, _ = pearsonr(p, y)
    return float(r)

def _ece_mce(y: np.ndarray, p: np.ndarray, n_bins: int = 15) -> tuple[float, float]:
    """
    Expected Calibration Error (ECE) and Max Calibration Error (MCE).
    Bins are linear in [0,1]. Returns (ECE, MCE).
    """
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    # Guard: if predictions are not in [0,1], clip (your loader already clamps)
    p = np.clip(p, 0.0, 1.0)
    N = len(p)
    if N == 0:
        return np.nan, np.nan

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    mce = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        mask = (p >= lo) & (p < hi) if b < n_bins - 1 else (p >= lo) & (p <= hi)
        nb = int(mask.sum())
        if nb == 0:
            continue
        conf = float(p[mask].mean())
        acc = float(y[mask].mean())  # observed frequency
        gap = abs(acc - conf)
        ece += (nb / N) * gap
        mce = max(mce, gap)
    return float(ece), float(mce)

def _f1_from_counts(tp, fp, fn):
    denom = (2*tp + fp + fn)
    return 0.0 if denom == 0 else (2*tp) / denom

def _metrics_at_threshold(scores: np.ndarray, labels: np.ndarray, thr: float):
    """Compute confusion + metrics at score >= thr => positive."""
    y_pred = (scores >= thr)
    y_true = labels.astype(bool)
    tp = int(np.sum(y_pred & y_true))
    fp = int(np.sum(y_pred & ~y_true))
    fn = int(np.sum(~y_pred & y_true))
    tn = int(np.sum(~y_pred & ~y_true))
    acc = (tp + tn) / max(1, len(labels))
    f1 = _f1_from_counts(tp, fp, fn)
    return tp, fp, fn, tn, acc, f1

def _binary_confusion(y_true: np.ndarray, y_prob: np.ndarray, thr: float) -> Dict[str, float]:
    y_pred = (y_prob >= thr).astype(np.uint8)
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    iou = tp / max(tp + fp + fn, 1)
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    tnr = tn / max(tn + fp, 1)  # specificity
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1, iou=iou, accuracy=acc, specificity=tnr)

def _mcc_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    """Matthews Correlation Coefficient from confusion counts."""
    num = (tp * tn) - (fp * fn)
    den = np.sqrt(max((tp + fp), 1) * max((tp + fn), 1) * max((tn + fp), 1) * max((tn + fn), 1))
    return 0.0 if den == 0 else float(num / den)

def _cohen_kappa_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    """Cohen's kappa from confusion counts."""
    total = tp + fp + fn + tn
    if total == 0:
        return float('nan')
    po = (tp + tn) / total
    # expected agreement by chance from marginals
    p_yes_pred = (tp + fp) / total
    p_yes_true = (tp + fn) / total
    p_no_pred  = (tn + fn) / total
    p_no_true  = (tn + fp) / total
    pe = p_yes_pred * p_yes_true + p_no_pred * p_no_true
    denom = (1.0 - pe)
    return 0.0 if denom == 0 else float((po - pe) / denom)

sort_by = 'Correlation'
def rank_and_test_models(stats_list, labels=None, alternative="two-sided", n_bins: int = 15):
    n = len(stats_list)
    if labels is None:
        labels = [f"model_{i}" for i in range(n)]
    else:
        # Normalize labels length to n to avoid IndexError
        if len(labels) < n:
            labels = labels + [f"model_{i}" for i in range(len(labels), n)]
        elif len(labels) > n:
            labels = labels[:n]
    """
    Parameters
    ----------
    stats_list : list[dict]
        Each element is a stats dict from compute_burn_control_stats *with per-image outputs*:
          - For burned_control: "burned_minus_control_per_image" (preferred), OR
            ("burned_means_per_image" and "control_means_per_image" as fallback).
          - For masked_unmasked: "masked_means_per_image" and "unmasked_means_per_image".
    labels : list[str] or None
        Optional human-readable names for the models. If None, uses f"model_{i}".
    alternative : {"two-sided","less","greater"}
        Alternative hypothesis for Mann–Whitney tests.
        Example: to test if row model < column model, use "less".

    Returns
    -------
    result : dict
        {
          "burned_control": {
              "order": [indices sorted by mean delta desc],
              "labels_sorted": [...],
              "mean_deltas_sorted": [...],
              "pairwise": {"U":..., "p":..., "AUC":..., "n_x":..., "n_y":...}
          },
          "masked_unmasked": {
              "order": ...,
              "labels_sorted": ...,
              "mean_deltas_sorted": ...,
              "pairwise": {...}
          }
        }
    """
    if labels is None:
        labels = [f"model_{i}" for i in range(len(stats_list))]


    def _prepare(mode):
        deltas = [_extract_deltas(s, mode) for s in stats_list]
        means = [float(np.mean(d)) if len(d) else float("nan") for d in deltas]
        order = np.argsort(means)[::-1]  # descending by mean delta
        pairwise = _pairwise_mw(deltas, alternative=alternative)

        metrics = {
            "Correlation": [],
            "Brier": [],
            "LogLoss": [],
            "ROC_AUC": [],
            "ECE": [],
            "Reliability(MCE)": [],
            "Delta": [],
            "Precision": [],
            "Recall": [],
            "f1": [],
            "IoU": [],
            "accuracy": [],
            "specificity": [],
            "mcc": [],
            "cohen_kappa": [],
        }

        for s in stats_list:
            y = np.asarray(s["labels"], dtype=float)
            p = np.asarray(s["preds"], dtype=float)
            r = _pearson_corr(y, p)
            br = _brier(y, p)
            ll = _logloss(y, p)
            auc = _roc_auc_from_scores(y>.5, p)
            ece, mce = _ece_mce(y, p, n_bins=n_bins)
            # Confusion-based metrics @ threshold
            tp, fp, fn, tn, acc, f1 = _metrics_at_threshold(p, y>.5, .5)
            cm = _binary_confusion(y>.5, p, .5)
            mcc = _mcc_from_counts(tp, fp, fn, tn)
            kappa = _cohen_kappa_from_counts(tp, fp, fn, tn)

            metrics["Delta"].append(s["diff_burned_minus_control"])
            metrics["Correlation"].append(r)
            metrics["Brier"].append(br)
            metrics["LogLoss"].append(ll)
            metrics["ROC_AUC"].append(auc)
            metrics["ECE"].append(ece)
            metrics["Reliability(MCE)"].append(mce)
            metrics["Precision"].append(float(cm["precision"]))
            metrics["Recall"].append(float(cm["recall"]))
            metrics["f1"].append(float(cm["f1"]))
            metrics["IoU"].append(float(cm["iou"]))
            metrics["accuracy"].append(float(cm["accuracy"]))
            metrics["specificity"].append(float(cm["specificity"]))
            metrics["mcc"].append(float(mcc))
            metrics["cohen_kappa"].append(float(kappa))

        # Determine best per column (higher-is-better vs lower-is-better)
        higher_better = {"Correlation", "ROC_AUC", "Delta", "Precision", "Recall", "f1", "IoU",
                        "accuracy", "specificity", "mcc", "cohen_kappa"}
        lower_better  = {"Brier", "LogLoss", "ECE", "Reliability(MCE)", "recon", "ssim", "grad", "mse", "mae"}
        col_names = list(metrics.keys())

        # Compute best values
        best_vals = {}
        for col in col_names:
            vals = np.array(metrics[col], dtype=float)
            if col in higher_better:
                best = np.nanmax(vals)
            else:
                best = np.nanmin(vals)
            best_vals[col] = best

        # Pretty print with bold best; ANSI bold if TTY, else **markdown**
        use_ansi = hasattr(sys.stdout, "isatty") and sys.stdout.isatty()
        def bold(s: str) -> str:
            return f"\033[1m{s}\033[0m" if use_ansi else f"**{s}**"

        # Order based in Reliability
        order = np.argsort(metrics[sort_by])
        if sort_by in higher_better:
            order = order[::-1]

        # Column widths
        model_w = max(5, max(len(m) for m in labels)) + 2
        col_w = 12

        # Header
        header = f"{'Model':<{model_w}}"
        for col in col_names:
            header += f"{col:>{col_w}}"
        print("\n=== Metrics (burned vs control, per-image mean as prediction) ===")
        print(header)

        # Rows
        for i, m in zip(order, np.array(labels)[order]):
            row = f"{m:<{model_w}}"
            for col in col_names:
                val = metrics[col][i]
                # format
                if np.isnan(val):
                    s = "nan"
                else:
                    s = f"{val:.3f}"
                # bold if best (with a tiny tolerance for ties)
                tol = 1e-12
                is_best = (
                    (col in higher_better and np.isfinite(val) and (best_vals[col] - val) <= tol) or
                    (col in lower_better and np.isfinite(val) and (val - best_vals[col]) <= tol)
                )
                row += f"{bold(s) if is_best else s:>{col_w}}"
            print(row)

        # Also return the raw numbers if you want to save/export later
        out = {m: {col: float(metrics[col][i]) for col in col_names} for i, m in enumerate(labels)}

        return {
            "order": order.tolist(),
            "labels_sorted": [labels[i] for i in order],
            "mean_deltas_sorted": [means[i] for i in order],
            "pairwise": pairwise,
            "metrics_out": out
        }

    return {
        "burned_control": _prepare("burned_control"),
    }


In [13]:
nocot = compute_burn_control_stats(f"{DATA_DIR}/vlm_forward_pass_predictions.json")
cot = compute_burn_control_stats(f"{DATA_DIR}/vlm_cot_predictions.json")
climate = compute_burn_control_stats(f"{DATA_DIR}/climate_predictions.json")
gpt = compute_burn_control_stats(f"{DATA_DIR}/gpt_predictions.json")
random = compute_burn_control_stats(f"{DATA_DIR}/random_predictions.json")

# -------------------------
# Example usage (after you computed stats0, stats2, ...):
res = rank_and_test_models(
    [gpt, nocot, cot, climate, random],
    labels=["gpt", "vlm_no_cot", "vlm_cot", "climate", "random"],
    alternative="two-sided")
print("Order (burned-control):", res["burned_control"]["labels_sorted"])
#Access pairwise p-values matrix:
p_bc = res["burned_control"]["pairwise"]["p"]

In [14]:
df = pd.DataFrame(res['burned_control']['metrics_out']).T.sort_values('Correlation', ascending=False)
def style_bold_best(df: pd.DataFrame, precision=4):
    """
    Bold the best value per column:
      - For 'higher is better' metrics: bold the max.
      - For 'lower is better' metrics: bold the min.
    """
    higher_is_better = {"Correlation", "ROC_AUC", "Delta", "Precision", "Recall", "f1", "IoU",
                    "accuracy", "specificity", "mcc", "cohen_kappa"}
    lower_is_better  = {"Brier", "LogLoss", "ECE", "Reliability(MCE)", "recon", "ssim", "grad", "mse", "mae"}

    def highlight(col: pd.Series) -> list:
        if col.name in higher_is_better:
            best = col.max(skipna=True)
            return ["font-weight: bold" if v == best else "" for v in col]
        elif col.name in lower_is_better:
            best = col.min(skipna=True)
            return ["font-weight: bold" if v == best else "" for v in col]
        else:
            return [""] * len(col)

    return (
        df.style
          .apply(highlight, axis=0)
          .format(precision=precision)
    )
display(style_bold_best(df, precision=4))

# Europe

In [19]:
import glob
import numpy as np
from tqdm import tqdm
from scipy.stats import mannwhitneyu
import os, sys
import pandas as pd
project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from config import DATA_DIR

def compute_burn_control_stats(json_path, fwi=True):
    df = pd.read_json(json_path).T

    # Accumulators
    burned_sum_all, burned_count_all = 0.0, 0
    control_sum_all, control_count_all = 0.0, 0
    burned_means_per_image = []
    control_means_per_image = []

    # 1) Burned predictions: global mean + masked/unmasked means (with masks)
    for lat_lon, row in tqdm(df[df.img_path.apply(lambda f: '/fire' in f)].iterrows(), desc="Burned preds"):
        pred = float(row['answer'])
        burned_sum_all += pred
        burned_count_all += 1
        burned_means_per_image.append(pred)

    # 2) Random control predictions: global mean
    for lat_lon, row in tqdm(df[df.img_path.apply(lambda f: '/no_fire' in f)].iterrows(), desc="Control preds"):
        pred = float(row['answer'])
        control_sum_all += pred
        control_count_all += 1
        control_means_per_image.append(pred)

    # Safeguards
    if burned_count_all == 0 or control_count_all == 0:
        raise ValueError("No pixels found in one of the sets.")

    # Means
    burned_mean_all = burned_sum_all / burned_count_all
    control_mean_all = control_sum_all / control_count_all
    diff_all = burned_mean_all - control_mean_all

    # Print results
    print("\n=== Averages ===")
    print(f"Burned predictions (all pixels): {burned_mean_all:.6f}")
    print(f"Random controls  (all pixels):   {control_mean_all:.6f}")
    print(f"Difference (burned - control):   {diff_all:.6f}")

    # Build predictions/labels (per-image level)
    preds = np.array(burned_means_per_image + control_means_per_image, dtype=float)/10.
    labels = np.concatenate([
        np.ones(len(burned_means_per_image), dtype=int),
        np.zeros(len(control_means_per_image), dtype=int)
    ])

    return {
        "burned_mean_all": burned_mean_all,
        "control_mean_all": control_mean_all,
        "diff_burned_minus_control": diff_all,
        "burned_means_per_image": burned_means_per_image,
        "control_means_per_image": control_means_per_image,
        # New keys for metrics:
        "preds": preds,     # per-image mean, in [0,1]
        "labels": labels,   # 1 for burned, 0 for control
    }

import numpy as np
from itertools import combinations
from scipy.stats import mannwhitneyu

def _safe_finite(arr):
    arr = np.asarray(arr, dtype=float)
    return arr[np.isfinite(arr)]

def _extract_deltas(stats, mode):
    """
    Return a per-image delta array for a given stats dict and mode:
      - mode == "burned_control": burned − control (per image)
      - mode == "masked_unmasked": masked − unmasked (per image)

    Requires per-image arrays returned by your modified compute_burn_control_stats:
      - "burned_minus_control_per_image"  (preferred for burned_control)
        OR fallback to difference of the per-image means if both lists exist
        ("burned_means_per_image", "control_means_per_image").
      - "masked_means_per_image" & "unmasked_means_per_image" for masked_unmasked.
    """
    if mode == "burned_control":
        if "burned_minus_control_per_image" in stats:
            return _safe_finite(stats["burned_minus_control_per_image"])
        # Fallback if unmatched controls: difference of per-image means (not ideal but usable)
        if ("burned_means_per_image" in stats) and ("control_means_per_image" in stats):
            b = _safe_finite(stats["burned_means_per_image"])
            c = _safe_finite(stats["control_means_per_image"])
            # Lengths may differ if controls are unmatched; compute all-vs-all deltas via broadcasting
            # but that explodes size. Instead, compare distributions with MW directly downstream.
            # For a delta distribution fallback, we take elementwise up to min length.
            n = min(len(b), len(c))
            if n == 0:
                return np.array([], dtype=float)
            return b[:n] - c[:n]
        raise ValueError("Stats object lacks per-image fields for burned_control comparison.")
    else:
        raise ValueError(f"Unknown mode: {mode}")

def _pairwise_mw(deltas_list, alternative="two-sided"):
    """
    Given a list of 1D arrays (per model), compute pairwise Mann–Whitney U tests.
    Returns a dict-of-dicts with entries: {"U": U, "p": p, "AUC": auc, "n_x": n1, "n_y": n2}.
    AUC (probability of superiority) = U / (n1*n2).
    """
    k = len(deltas_list)
    U = [[None]*k for _ in range(k)]
    P = [[None]*k for _ in range(k)]
    A = [[None]*k for _ in range(k)]
    N1 = [[None]*k for _ in range(k)]
    N2 = [[None]*k for _ in range(k)]
    for i, j in combinations(range(k), 2):
        x = _safe_finite(deltas_list[i])
        y = _safe_finite(deltas_list[j])
        if len(x) == 0 or len(y) == 0:
            u = p = auc = np.nan
        else:
            u, p = mannwhitneyu(x, y, alternative=alternative)
            auc = u / (len(x)*len(y))
        U[i][j] = U[j][i] = u
        P[i][j] = P[j][i] = p
        A[i][j] = A[j][i] = auc
        N1[i][j] = N1[j][i] = len(x)
        N2[i][j] = N2[j][i] = len(y)
        # Diagonal
    for i in range(k):
        U[i][i] = P[i][i] = A[i][i] = 0.0
        N1[i][i] = N2[i][i] = len(_safe_finite(deltas_list[i]))
    return {"U": U, "p": P, "AUC": A, "n_x": N1, "n_y": N2}


# ----------------------- Metrics & summary table -----------------------
from scipy.stats import mannwhitneyu, pearsonr

def _roc_auc_from_scores(y: np.ndarray, p: np.ndarray) -> float:
    """ROC AUC via Mann–Whitney: P(p_pos > p_neg)."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pos = p[y == 1]
    neg = p[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    U = mannwhitneyu(pos, neg, alternative="greater", method="asymptotic").statistic
    return float(U / (len(pos) * len(neg)))

def _brier(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    return float(np.mean((p - y) ** 2))

def _logloss(y: np.ndarray, p: np.ndarray, eps: float = 1e-12) -> float:
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    p = np.clip(p, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))

def _pearson_corr(y: np.ndarray, p: np.ndarray) -> float:
    # Point-biserial correlation equals Pearson between binary y and continuous p
    if len(y) < 2 or np.all(y == y[0]):
        return np.nan
    r, _ = pearsonr(p, y)
    return float(r)

def _ece_mce(y: np.ndarray, p: np.ndarray, n_bins: int = 15) -> tuple[float, float]:
    """
    Expected Calibration Error (ECE) and Max Calibration Error (MCE).
    Bins are linear in [0,1]. Returns (ECE, MCE).
    """
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    # Guard: if predictions are not in [0,1], clip (your loader already clamps)
    p = np.clip(p, 0.0, 1.0)
    N = len(p)
    if N == 0:
        return np.nan, np.nan

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    mce = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        mask = (p >= lo) & (p < hi) if b < n_bins - 1 else (p >= lo) & (p <= hi)
        nb = int(mask.sum())
        if nb == 0:
            continue
        conf = float(p[mask].mean())
        acc = float(y[mask].mean())  # observed frequency
        gap = abs(acc - conf)
        ece += (nb / N) * gap
        mce = max(mce, gap)
    return float(ece), float(mce)

def _f1_from_counts(tp, fp, fn):
    denom = (2*tp + fp + fn)
    return 0.0 if denom == 0 else (2*tp) / denom

def _metrics_at_threshold(scores: np.ndarray, labels: np.ndarray, thr: float):
    """Compute confusion + metrics at score >= thr => positive."""
    y_pred = (scores >= thr)
    y_true = labels.astype(bool)
    tp = int(np.sum(y_pred & y_true))
    fp = int(np.sum(y_pred & ~y_true))
    fn = int(np.sum(~y_pred & y_true))
    tn = int(np.sum(~y_pred & ~y_true))
    acc = (tp + tn) / max(1, len(labels))
    f1 = _f1_from_counts(tp, fp, fn)
    return tp, fp, fn, tn, acc, f1

def _binary_confusion(y_true: np.ndarray, y_prob: np.ndarray, thr: float) -> Dict[str, float]:
    y_pred = (y_prob >= thr).astype(np.uint8)
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    iou = tp / max(tp + fp + fn, 1)
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    tnr = tn / max(tn + fp, 1)  # specificity
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1, iou=iou, accuracy=acc, specificity=tnr)

def _mcc_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    """Matthews Correlation Coefficient from confusion counts."""
    num = (tp * tn) - (fp * fn)
    den = np.sqrt(max((tp + fp), 1) * max((tp + fn), 1) * max((tn + fp), 1) * max((tn + fn), 1))
    return 0.0 if den == 0 else float(num / den)

def _cohen_kappa_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    """Cohen's kappa from confusion counts."""
    total = tp + fp + fn + tn
    if total == 0:
        return float('nan')
    po = (tp + tn) / total
    # expected agreement by chance from marginals
    p_yes_pred = (tp + fp) / total
    p_yes_true = (tp + fn) / total
    p_no_pred  = (tn + fn) / total
    p_no_true  = (tn + fp) / total
    pe = p_yes_pred * p_yes_true + p_no_pred * p_no_true
    denom = (1.0 - pe)
    return 0.0 if denom == 0 else float((po - pe) / denom)

sort_by = 'Correlation'
def rank_and_test_models(stats_list, labels=None, alternative="two-sided", n_bins: int = 15):
    n = len(stats_list)
    if labels is None:
        labels = [f"model_{i}" for i in range(n)]
    else:
        # Normalize labels length to n to avoid IndexError
        if len(labels) < n:
            labels = labels + [f"model_{i}" for i in range(len(labels), n)]
        elif len(labels) > n:
            labels = labels[:n]
    """
    Parameters
    ----------
    stats_list : list[dict]
        Each element is a stats dict from compute_burn_control_stats *with per-image outputs*:
          - For burned_control: "burned_minus_control_per_image" (preferred), OR
            ("burned_means_per_image" and "control_means_per_image" as fallback).
          - For masked_unmasked: "masked_means_per_image" and "unmasked_means_per_image".
    labels : list[str] or None
        Optional human-readable names for the models. If None, uses f"model_{i}".
    alternative : {"two-sided","less","greater"}
        Alternative hypothesis for Mann–Whitney tests.
        Example: to test if row model < column model, use "less".

    Returns
    -------
    result : dict
        {
          "burned_control": {
              "order": [indices sorted by mean delta desc],
              "labels_sorted": [...],
              "mean_deltas_sorted": [...],
              "pairwise": {"U":..., "p":..., "AUC":..., "n_x":..., "n_y":...}
          },
          "masked_unmasked": {
              "order": ...,
              "labels_sorted": ...,
              "mean_deltas_sorted": ...,
              "pairwise": {...}
          }
        }
    """
    if labels is None:
        labels = [f"model_{i}" for i in range(len(stats_list))]


    def _prepare(mode):
        deltas = [_extract_deltas(s, mode) for s in stats_list]
        means = [float(np.mean(d)) if len(d) else float("nan") for d in deltas]
        order = np.argsort(means)[::-1]  # descending by mean delta
        pairwise = _pairwise_mw(deltas, alternative=alternative)

        metrics = {
            "Correlation": [],
            "Brier": [],
            "LogLoss": [],
            "ROC_AUC": [],
            "ECE": [],
            "Reliability(MCE)": [],
            "Delta": [],
            "Precision": [],
            "Recall": [],
            "f1": [],
            "IoU": [],
            "accuracy": [],
            "specificity": [],
            "mcc": [],
            "cohen_kappa": [],
        }

        for s in stats_list:
            y = np.asarray(s["labels"], dtype=int)
            p = np.asarray(s["preds"], dtype=float)
            r = _pearson_corr(y, p)
            br = _brier(y, p)
            ll = _logloss(y, p)
            auc = _roc_auc_from_scores(y, p)
            ece, mce = _ece_mce(y, p, n_bins=n_bins)
            # Confusion-based metrics @ threshold
            tp, fp, fn, tn, acc, f1 = _metrics_at_threshold(p, y, .5)
            cm = _binary_confusion(y, p, .5)
            mcc = _mcc_from_counts(tp, fp, fn, tn)
            kappa = _cohen_kappa_from_counts(tp, fp, fn, tn)

            metrics["Delta"].append(s["diff_burned_minus_control"])
            metrics["Correlation"].append(r)
            metrics["Brier"].append(br)
            metrics["LogLoss"].append(ll)
            metrics["ROC_AUC"].append(auc)
            metrics["ECE"].append(ece)
            metrics["Reliability(MCE)"].append(mce)
            metrics["Precision"].append(float(cm["precision"]))
            metrics["Recall"].append(float(cm["recall"]))
            metrics["f1"].append(float(cm["f1"]))
            metrics["IoU"].append(float(cm["iou"]))
            metrics["accuracy"].append(float(cm["accuracy"]))
            metrics["specificity"].append(float(cm["specificity"]))
            metrics["mcc"].append(float(mcc))
            metrics["cohen_kappa"].append(float(kappa))

        # Determine best per column (higher-is-better vs lower-is-better)
        higher_better = {"Correlation", "ROC_AUC", "Delta", "Precision", "Recall", "f1", "IoU",
                        "accuracy", "specificity", "mcc", "cohen_kappa"}
        lower_better  = {"Brier", "LogLoss", "ECE", "Reliability(MCE)", "recon", "ssim", "grad", "mse", "mae"}
        col_names = list(metrics.keys())

        # Compute best values
        best_vals = {}
        for col in col_names:
            vals = np.array(metrics[col], dtype=float)
            if col in higher_better:
                best = np.nanmax(vals)
            else:
                best = np.nanmin(vals)
            best_vals[col] = best

        # Pretty print with bold best; ANSI bold if TTY, else **markdown**
        use_ansi = hasattr(sys.stdout, "isatty") and sys.stdout.isatty()
        def bold(s: str) -> str:
            return f"\033[1m{s}\033[0m" if use_ansi else f"**{s}**"

        # Order based in Reliability
        order = np.argsort(metrics[sort_by])
        if sort_by in higher_better:
            order = order[::-1]

        # Column widths
        model_w = max(5, max(len(m) for m in labels)) + 2
        col_w = 12

        # Header
        header = f"{'Model':<{model_w}}"
        for col in col_names:
            header += f"{col:>{col_w}}"
        print("\n=== Metrics (burned vs control, per-image mean as prediction) ===")
        print(header)

        # Rows
        for i, m in zip(order, np.array(labels)[order]):
            row = f"{m:<{model_w}}"
            for col in col_names:
                val = metrics[col][i]
                # format
                if np.isnan(val):
                    s = "nan"
                else:
                    s = f"{val:.3f}"
                # bold if best (with a tiny tolerance for ties)
                tol = 1e-12
                is_best = (
                    (col in higher_better and np.isfinite(val) and (best_vals[col] - val) <= tol) or
                    (col in lower_better and np.isfinite(val) and (val - best_vals[col]) <= tol)
                )
                row += f"{bold(s) if is_best else s:>{col_w}}"
            print(row)

        # Also return the raw numbers if you want to save/export later
        out = {m: {col: float(metrics[col][i]) for col in col_names} for i, m in enumerate(labels)}

        return {
            "order": order.tolist(),
            "labels_sorted": [labels[i] for i in order],
            "mean_deltas_sorted": [means[i] for i in order],
            "pairwise": pairwise,
            "metrics_out": out
        }

    return {
        "burned_control": _prepare("burned_control"),
    }


In [20]:
nocot = compute_burn_control_stats(f"{DATA_DIR}/vlm_forward_pass_predictions.json")
cot = compute_burn_control_stats(f"{DATA_DIR}/vlm_cot_predictions.json")
climate = compute_burn_control_stats(f"{DATA_DIR}/climate_predictions.json")
# q_climate = compute_burn_control_stats(f"{DATA_DIR}/climate-answers_predictions.json")
gpt = compute_burn_control_stats(f"{DATA_DIR}/gpt_predictions_full.json")
fwi = compute_burn_control_stats(f"{DATA_DIR}/fwi-mean-jjas_predictions.json")
fwi_n = compute_burn_control_stats(f"{DATA_DIR}/fwi-mean-jjas_predictions_normed.json")
# q_fwi = compute_burn_control_stats(f"{DATA_DIR}/fwi-answers_predictions.json")
random = compute_burn_control_stats(f"{DATA_DIR}/random_predictions.json")

# -------------------------
# Example usage (after you computed stats0, stats2, ...):
res = rank_and_test_models(
    [gpt, nocot, cot, climate, fwi, fwi_n, random],
    labels=["gpt", "vlm_no_cot", "vlm_cot", "climate", "fwi", "fwi_n", "random"],
    alternative="two-sided")
print("Order (burned-control):", res["burned_control"]["labels_sorted"])
#Access pairwise p-values matrix:
p_bc = res["burned_control"]["pairwise"]["p"]

In [22]:
df = pd.DataFrame(res['burned_control']['metrics_out']).T.sort_values('Correlation', ascending=False)
def style_bold_best(df: pd.DataFrame, precision=6):
    """
    Bold the best value per column:
      - For 'higher is better' metrics: bold the max.
      - For 'lower is better' metrics: bold the min.
    """
    higher_is_better = {"Correlation", "ROC_AUC", "Delta", "Precision", "Recall", "f1", "IoU",
                    "accuracy", "specificity", "mcc", "cohen_kappa"}
    lower_is_better  = {"Brier", "LogLoss", "ECE", "Reliability(MCE)", "recon", "ssim", "grad", "mse", "mae"}

    def highlight(col: pd.Series) -> list:
        if col.name in higher_is_better:
            best = col.max(skipna=True)
            return ["font-weight: bold" if v == best else "" for v in col]
        elif col.name in lower_is_better:
            best = col.min(skipna=True)
            return ["font-weight: bold" if v == best else "" for v in col]
        else:
            return [""] * len(col)

    return (
        df.style
          .apply(highlight, axis=0)
          .format(precision=precision)
    )
display(style_bold_best(df, precision=4))

In [25]:
import numpy as np

def _bh_fdr(pvals, alpha=0.05):
    """Benjamini–Hochberg correction; returns boolean 'significant' mask in flat order."""
    p = np.array(pvals, dtype=float)
    m = p.size
    order = np.argsort(p)
    ranked = p[order]
    thresh = alpha * (np.arange(1, m+1) / m)
    passed = ranked <= np.maximum.accumulate(thresh)
    # mark only up to the largest k that passed
    k = np.where(passed)[0].max()+1 if passed.any() else 0
    sig = np.zeros_like(p, dtype=bool)
    if k > 0:
        sig[order[:k]] = True
    return sig
def show_pairwise(res, section="burned_control", alpha=0.05, fdr=False):
    labels_sorted = res[section]["labels_sorted"]
    order = res[section]["order"]
    P = np.array(res[section]["pairwise"]["p"])
    A = np.array(res[section]["pairwise"]["AUC"])

    # reorder to match displayed order
    P = P[np.ix_(order, order)]
    A = A[np.ix_(order, order)]

    # optional FDR correction
    sig = None
    if fdr:
        tri_idx = np.triu_indices_from(P, k=1)
        sig_flat = _bh_fdr(P[tri_idx], alpha=alpha)
        sig = np.zeros_like(P, dtype=bool)
        sig[tri_idx] = sig_flat

    print(f"\n=== {section} ===")
    print("Order:", labels_sorted)
    print("Mean deltas (desc):", [round(x, 6) for x in res[section]["mean_deltas_sorted"]])

    # Set consistent column width
    colw = max(max(len(l) for l in labels_sorted), 8) + 2

    def _fmt(val, sigmark=False):
        if np.isnan(val):
            s = "nan"
        else:
            s = f"{val:.3g}"
        if sigmark:
            s += "*"
        return s

    # Header
    print("\nPairwise p-values (rows vs cols; alternative set in rank_and_test_models):")
    print("".ljust(colw) + "".join(f"{l:<{colw}}" for l in labels_sorted))
    for i, rlab in enumerate(labels_sorted):
        row = f"{rlab:<{colw}}"
        for j in range(len(labels_sorted)):
            smark = bool(sig is not None and sig[i, j])
            row += f"{_fmt(P[i, j], smark):<{colw}}"
        print(row)

    print("\nPairwise AUC (probability row > col):")
    print("".ljust(colw) + "".join(f"{l:<{colw}}" for l in labels_sorted))
    for i, rlab in enumerate(labels_sorted):
        row = f"{rlab:<{colw}}"
        for j in range(len(labels_sorted)):
            row += f"{A[i, j]:<{colw}.3f}"
        print(row)
print("oracles evaluation")
show_pairwise(res, "burned_control", fdr=True)